# Практика 04. Нейросеть на NumPy

**Версия:** 2026-09-25 (f95bbe8)

**Как сдавать работу**

1. Откройте ноутбук в Colab и сохраните копию себе: «Файл → Сохранить копию на Диске». Работайте в копии.
2. Выполните задания: код пишите вместо `# ВАШ КОД ЗДЕСЬ` / `raise NotImplementedError`,
   ответы на вопросы — после «*Ваш ответ:*».
3. После каждого задания запускайте ячейку с проверками. Проверки — для самоконтроля:
   их прохождение не гарантирует зачёт, а текстовые ответы проверяются отдельно.
4. Перед сдачей выполните «Среда выполнения → Перезапустить сеанс и выполнить все»: ноутбук должен
   выполниться целиком без ошибок.
5. Откройте доступ по ссылке («Настройки доступа → Все, у кого есть ссылка») и вставьте ссылку
   на свою копию в таблицу курса.

Свёрнутые ячейки со значком ▶ — служебные (загрузка данных, функции проверки). Их нужно выполнять,
но менять не нужно.

К лекции 03. План работы:

1. **Часть 1** — полносвязная сеть своими руками на NumPy: активации, softmax и кросс-энтропия, инициализация,
   прямой и обратный проход, обучение мини-батчами. Прямой проход сверяется с `MLPClassifier` из scikit-learn,
   обратный — с численным градиентом.
2. **Часть 2** — `MLPClassifier` на тех же данных: подбор архитектуры и регуляризации, сравнение со своей сетью и с
   логистической регрессией.
3. **Часть 3** — эксперимент и выводы: ширина сети, шаг обучения, регуляризация, нулевая инициализация. Код здесь
   простой, оценивается объяснение.

**Соглашения в коде.** Объекты — строки матрицы: мини-батч $\mathbf{X} \in \mathbb{R}^{m \times d}$. Веса слоя хранятся как
матрица формы `(d_in, d_out)`, прямой проход слоя — `Z = H @ W + b`. В лекции векторы — столбцы и веса
транспонированы; это та же сеть. Параметры сети — список пар `(W, b)` по слоям.

В заданиях части 1, где сказано «без циклов», циклы по объектам запрещены; цикл по слоям нужен.

In [ ]:
# @title Служебная ячейка: импорты, данные и функции проверки { display-mode: "form" }
import inspect
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
SEED = 0
rng = np.random.default_rng(SEED)


def assert_no_loops(func):
    """Проверяет, что в теле функции нет циклов for/while (включения списков тоже считаются)."""
    source = inspect.getsource(func)
    body = source.split('"""')[-1] if '"""' in source else source
    assert not re.search(r"\b(for|while)\b", body), (
        f"В функции {func.__name__} есть цикл. Здесь нужно решение без циклов — операциями над матрицами."
    )


def make_spirals(n_per_class=200, n_classes=3, noise=0.5, seed=SEED):
    """Спирали: n_classes закрученных «рукавов» на плоскости; noise — разброс угла."""
    r = np.random.default_rng(seed)
    X, y = [], []
    for k in range(n_classes):
        t = np.linspace(0.05, 1, n_per_class)
        angle = 4 * t + 2 * np.pi * k / n_classes + r.normal(scale=noise, size=n_per_class)
        X.append(np.c_[t * np.cos(angle), t * np.sin(angle)])
        y.append(np.full(n_per_class, k))
    return np.vstack(X), np.concatenate(y)


def one_hot(y, K):
    return np.eye(K)[y]


def plot_boundary(predict, X, y, ax=None, title=""):
    """Области классов, которые предсказывает функция predict(X) -> метки."""
    ax = ax or plt.gca()
    xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 300), np.linspace(-1.2, 1.2, 300))
    zz = predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.3, levels=[-0.5, 0.5, 1.5, 2.5], cmap="brg")
    ax.scatter(X[:, 0], X[:, 1], c=y, s=8, cmap="brg", edgecolors="k", linewidths=0.2)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


from sklearn.model_selection import train_test_split

X_all, y_all = make_spirals()
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.3, random_state=SEED, stratify=y_all)
K = 3
plt.figure(figsize=(4, 4))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=8, cmap="brg")
plt.title("Обучающая выборка: три спирали")
plt.show()

# Часть 1. Сеть своими руками

## Задание 1.1. Активации и softmax

Напишите поэлементные `relu(Z)` и её производную `relu_grad(Z)` (единица при $z > 0$, иначе ноль), а также
`softmax(Z)` по строкам. Экспонента больших чисел переполняется (`np.exp(1000)` — это `inf`), поэтому перед
экспонентой вычтите из каждой строки её максимум: softmax от этого не меняется, потому что
$\frac{e^{z_k - c}}{\sum_m e^{z_m - c}} = \frac{e^{z_k}}{\sum_m e^{z_m}}$. Без циклов.

In [ ]:
def relu(Z):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def relu_grad(Z):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def softmax(Z):
    """Softmax по строкам матрицы Z (m, K)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from scipy.special import softmax as sp_softmax

Z = rng.normal(scale=3, size=(6, 4))
assert np.array_equal(relu(np.array([-1.0, 0.0, 2.0])), [0, 0, 2]), "relu(-1, 0, 2) должна быть (0, 0, 2)"
assert np.array_equal(relu_grad(np.array([-1.0, 0.0, 2.0])), [0, 0, 1]), "Производная ReLU: 1 при z > 0, иначе 0"
assert np.allclose(softmax(Z), sp_softmax(Z, axis=1)), "softmax не совпадает с scipy.special.softmax по строкам"
big = softmax(np.array([[1000.0, 1001.0, 1002.0]]))
assert np.all(np.isfinite(big)), "softmax переполняется на больших числах: вычтите максимум строки"
assert np.allclose(big, sp_softmax([1000.0, 1001.0, 1002.0])), "softmax больших чисел посчитан неверно"
for f in (relu, relu_grad, softmax):
    assert_no_loops(f)
print("OK")

## Задание 1.2. Кросс-энтропия

Для матрицы вероятностей $\mathbf{P} \in \mathbb{R}^{m \times K}$ и one-hot меток $\mathbf{Y}$ функция потерь — средняя по
объектам кросс-энтропия
$$
\ell = -\frac{1}{m}\sum_{i=1}^m \sum_{k=1}^K Y_{ik} \log P_{ik}.
$$
Напишите `cross_entropy(P, Y)`. Если какая-то вероятность равна нулю, получится $0 \cdot \log 0 = $ `nan`,
поэтому перед логарифмом ограничьте вероятности снизу: `np.clip(P, 1e-12, 1)` (так же поступает sklearn). Без циклов.

In [ ]:
def cross_entropy(P, Y):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.metrics import log_loss

P = softmax(rng.normal(size=(10, 3)))
labels = rng.integers(0, 3, size=10)
assert np.isclose(cross_entropy(P, one_hot(labels, 3)), log_loss(labels, P, labels=[0, 1, 2])), "Кросс-энтропия не совпадает с sklearn.metrics.log_loss"
assert np.isclose(cross_entropy(np.array([[1.0, 0.0]]), np.array([[1.0, 0.0]])), 0), "Для идеального предсказания потери равны 0"
assert_no_loops(cross_entropy)
print("OK")

## Задание 1.3. Инициализация

Напишите `init_params(sizes, seed)`. `sizes` — размеры слоёв, например `[2, 64, 3]`: вход, скрытые слои, выход.
Для каждой пары соседних размеров `(d_in, d_out)` создайте веса формы `(d_in, d_out)` из нормального распределения
со средним 0 и стандартным отклонением $\sqrt{2 / d_{\text{in}}}$ (так называемая инициализация Хе — почему именно
так, обсудим в лекции 04) и нулевые смещения длины `d_out`. Используйте один генератор
`np.random.default_rng(seed)` для всех слоёв по порядку. Верните список пар `(W, b)`.

In [ ]:
def init_params(sizes, seed=SEED):
    """Список пар (W, b): W формы (d_in, d_out), b формы (d_out,)."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
params = init_params([2, 5, 4, 3], seed=1)
assert len(params) == 3, f"Для sizes=[2, 5, 4, 3] должно быть 3 слоя, получено {len(params)}"
assert [W.shape for W, b in params] == [(2, 5), (5, 4), (4, 3)], f"Неверные формы весов: {[W.shape for W, b in params]}"
assert all(np.array_equal(b, np.zeros(W.shape[1])) for W, b in params), "Смещения должны быть нулевыми векторами длины d_out"
assert all(np.array_equal(W1, W2) for (W1, _), (W2, _) in zip(params, init_params([2, 5, 4, 3], seed=1))), "При одном seed результат должен повторяться"
W_big, _ = init_params([500, 400], seed=2)[0]
assert abs(W_big.std() - np.sqrt(2 / 500)) < 0.05 * np.sqrt(2 / 500), f"Стандартное отклонение весов {W_big.std():.4f}, а должно быть sqrt(2/d_in) = {np.sqrt(2 / 500):.4f}"
assert abs(W_big.mean()) < 0.005, "Среднее весов должно быть около нуля"
print("OK")

## Задание 1.4. Прямой проход

Напишите `forward(params, X)`. Все слои, кроме последнего, — `Z = H @ W + b`, затем ReLU; последний — `Z = H @ W + b`,
затем softmax. Верните пару `(P, cache)`, где `P` — вероятности классов, а `cache` — список пар `(Z, H)` для
обратного прохода: `cache[0] = (None, X)`, далее для каждого слоя по порядку его `Z` и выход `H` (для последнего
слоя `H = P`). Без циклов по объектам.

In [ ]:
def forward(params, X):
    """Прямой проход. Возвращает (P, cache), cache = [(None, X), (Z1, H1), ..., (ZL, P)]."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError


def predict(params, X):
    """Метки классов — argmax вероятностей."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
from sklearn.neural_network import MLPClassifier

params = [(W, rng.normal(scale=0.5, size=b.shape)) for W, b in init_params([2, 5, 4, 3], seed=1)]  # ненулевые смещения
P, cache = forward(params, X_train)
assert P.shape == (len(X_train), 3) and np.allclose(P.sum(axis=1), 1), "P — матрица (m, 3) вероятностей, строки в сумме дают 1"
assert len(cache) == 4 and cache[0][0] is None and np.array_equal(cache[0][1], X_train), "cache = [(None, X), (Z1, H1), (Z2, H2), (Z3, P)]"
assert np.allclose(cache[1][1], np.maximum(cache[1][0], 0)), "Скрытые слои: H = relu(Z)"
# Сеть sklearn той же архитектуры, в которую подставлены наши веса, должна давать те же вероятности.
sk = MLPClassifier(hidden_layer_sizes=(5, 4), activation="relu", max_iter=1).fit(X_train, y_train)
sk.coefs_ = [W for W, b in params]
sk.intercepts_ = [b for W, b in params]
assert np.allclose(P, sk.predict_proba(X_train)), "Прямой проход не совпадает с MLPClassifier с теми же весами"
assert np.array_equal(predict(params, X_train), sk.predict(X_train)), "predict: argmax вероятностей"
print("OK")

## Задание 1.5. Обратный проход

Напишите `backward(params, cache, Y)` — градиенты **средней** по батчу кросс-энтропии по всем параметрам. Формулы
лекции в «строчной» записи, $m$ — размер батча:

- выходной слой: $\boldsymbol{\Delta}^{(L)} = \frac{1}{m}(\mathbf{P} - \mathbf{Y})$ (множитель $\frac{1}{m}$ удобно внести сразу);
- для слоя $l$, от последнего к первому: $\nabla_{\mathbf{W}^{(l)}} = \mathbf{H}^{(l-1)\top}\boldsymbol{\Delta}^{(l)}$,
  $\nabla_{\mathbf{b}^{(l)}}$ — сумма строк $\boldsymbol{\Delta}^{(l)}$;
- переход к предыдущему слою: $\boldsymbol{\Delta}^{(l-1)} = \bigl(\boldsymbol{\Delta}^{(l)}\mathbf{W}^{(l)\top}\bigr) \odot \mathrm{relu}'\bigl(\mathbf{Z}^{(l-1)}\bigr)$.

Верните список пар `(dW, db)` в том же порядке, что `params`. Без циклов по объектам.

In [ ]:
def backward(params, cache, Y):
    """Градиенты средней кросс-энтропии: список (dW, db) по слоям."""
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
def numerical_gradient(params, X, Y, eps=1e-6):
    """Численный градиент кросс-энтропии центральной разностью (медленно, только для проверки)."""
    grads = []
    for W, b in params:
        gW, gb = np.zeros_like(W), np.zeros_like(b)
        for arr, g in ((W, gW), (b, gb)):
            for idx in np.ndindex(arr.shape):
                old = arr[idx]
                arr[idx] = old + eps
                plus = cross_entropy(forward(params, X)[0], Y)
                arr[idx] = old - eps
                minus = cross_entropy(forward(params, X)[0], Y)
                arr[idx] = old
                g[idx] = (plus - minus) / (2 * eps)
        grads.append((gW, gb))
    return grads


params = init_params([2, 5, 4, 3], seed=3)
for W, b in params:
    b += rng.normal(scale=0.1, size=b.shape)  # ненулевые смещения — чтобы проверить и их градиенты
X_small, Y_small = X_train[:7], one_hot(y_train[:7], 3)
_, cache = forward(params, X_small)
grads = backward(params, cache, Y_small)
num = numerical_gradient(params, X_small, Y_small)
assert len(grads) == 3 and all(gW.shape == W.shape and gb.shape == b.shape for (gW, gb), (W, b) in zip(grads, params)), (
    "backward должна вернуть по паре (dW, db) на слой, формы — как у W и b"
)
for layer, ((gW, gb), (nW, nb)) in enumerate(zip(grads, num)):
    for name, a, n in (("dW", gW, nW), ("db", gb, nb)):
        rel = np.abs(a - n).max() / max(np.abs(a).max(), np.abs(n).max(), 1e-12)
        assert rel < 1e-5, f"Слой {layer + 1}, {name}: относительная ошибка градиента {rel:.1e} — проверьте формулы и транспонирования"
print("OK: аналитический градиент совпадает с численным")

## Задание 1.6. Обучение

Напишите `train(params, X, y, lr, epochs, batch_size, seed)`: мини-батчевый градиентный спуск. Создайте один
генератор `np.random.default_rng(seed)`; в начале каждой эпохи получите перестановку объектов
(`rng.permutation(len(X))`) и идите по ней кусками по `batch_size`. На каждом батче — прямой проход, обратный
проход и шаг `W -= lr * dW`, `b -= lr * db`. После каждой эпохи запишите в `history` кросс-энтропию на всей выборке
`X`. Верните `(params, history)`. Исходный список `params` можно не сохранять.

In [ ]:
def train(params, X, y, lr=0.5, epochs=100, batch_size=32, seed=SEED):
    # ВАШ КОД ЗДЕСЬ
    raise NotImplementedError

In [ ]:
net, history = train(init_params([2, 64, 64, 3], seed=SEED), X_train, y_train, lr=0.5, epochs=200, batch_size=32)
train_acc = np.mean(predict(net, X_train) == y_train)
assert len(history) == 200, f"history — по значению на эпоху, получено {len(history)}"
assert history[-1] < history[0] / 3, "Функция потерь должна заметно уменьшиться за 200 эпох"
assert train_acc > 0.9, f"Точность на обучении {train_acc:.3f} — сеть обучилась плохо"
print(f"OK: точность на обучении {train_acc:.3f}")
plot_boundary(lambda Z: predict(net, Z), X_train, y_train, title="Своя сеть 2-64-64-3")
plt.show()

# Часть 2. MLPClassifier из scikit-learn

## Задание 2.1. Подбор архитектуры и регуляризации

Подберите поиском по сетке конвейер `StandardScaler` + `MLPClassifier(max_iter=2000, random_state=SEED)`:
сетка — `PARAM_GRID` (размеры скрытых слоёв и коэффициент $L_2$-регуляризации `alpha`), кросс-валидация — `CV`,
метрика — accuracy (по умолчанию). Результат — `grid_mlp`, обученный на `X_train`.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PARAM_GRID = {
    "mlpclassifier__hidden_layer_sizes": [(8,), (64,), (64, 64)],
    "mlpclassifier__alpha": [1e-4, 1e-2, 1.0],
}
CV = StratifiedKFold(3, shuffle=True, random_state=SEED)

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

print("лучшие параметры:", grid_mlp.best_params_, f"accuracy (CV) = {grid_mlp.best_score_:.3f}")

In [ ]:
assert isinstance(grid_mlp, GridSearchCV) and len(grid_mlp.cv_results_["params"]) == 9, "Нужен GridSearchCV по всей сетке PARAM_GRID (9 комбинаций)"
assert grid_mlp.n_splits_ == 3, "Кросс-валидация — CV (3 фолда)"
assert "StandardScaler" in repr(grid_mlp.estimator), "В конвейере нужен StandardScaler"
assert grid_mlp.best_score_ > 0.85, f"accuracy на кросс-валидации подозрительно низкая: {grid_mlp.best_score_:.3f}"
print("OK")

## Задание 2.2. Сравнение моделей

Заполните словарь `results` точностью (accuracy) на **тестовой** выборке:

- `"logreg"` — `LogisticRegression()` на исходных признаках;
- `"numpy_net"` — ваша сеть `net` из задания 1.6;
- `"sklearn_mlp"` — лучшая модель `grid_mlp`.

In [ ]:
from sklearn.linear_model import LogisticRegression

# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

pd.Series(results, name="accuracy на тесте").round(3)

In [ ]:
assert set(results) == {"logreg", "numpy_net", "sklearn_mlp"}, f"Нужны ключи logreg, numpy_net, sklearn_mlp; получено {set(results)}"
assert np.isclose(results["numpy_net"], np.mean(predict(net, X_test) == y_test)), "numpy_net: точность своей сети на X_test"
assert np.isclose(results["sklearn_mlp"], grid_mlp.score(X_test, y_test)), "sklearn_mlp: точность grid_mlp на X_test"
assert results["logreg"] < 0.7, "Логистическая регрессия не должна справляться со спиралями — проверьте, что она обучена на исходных признаках"
assert results["numpy_net"] > 0.85, f"Точность своей сети на тесте {results['numpy_net']:.3f} — подозрительно низкая"
print("OK")

# Часть 3. Эксперимент и выводы

## Задание 3.1. Ширина скрытого слоя

Для каждого числа нейронов из `WIDTHS` обучите на `X_train` сеть с одним скрытым слоем:
`MLPClassifier(hidden_layer_sizes=(h,), solver="lbfgs", max_iter=5000, random_state=SEED)` (метод L-BFGS надёжно
доводит обучение маленьких сетей до конца). Сохраните точность на обучении и на тесте в списки `width_train` и
`width_test` и постройте их на одном графике (логарифмическая шкала по ширине). Постройте также границы решений
для нескольких ширин функцией `plot_boundary`.

In [ ]:
WIDTHS = [1, 2, 4, 8, 32, 128]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(width_train) == len(WIDTHS) and len(width_test) == len(WIDTHS), "По значению на каждую ширину"
assert width_test[0] < 0.7, "Сеть с одним нейроном не должна решать задачу"
assert max(width_test) > 0.9, "Широкая сеть должна решать задачу хорошо"
print("OK")

## Задание 3.2. Шаг обучения

Обучите свою сеть `[2, 64, 3]` функцией `train` (100 эпох, батч 32, одна и та же инициализация `init_params(..., seed=SEED)`)
с шагами из `LRS`. Сохраните истории в словарь `lr_history` (шаг → history) и постройте их на одном графике
(логарифмическая шкала по оси потерь). Большой шаг может дать `nan` и предупреждения о переполнении — это часть
эксперимента, предупреждения можно подавить (`np.errstate(all="ignore")`).

In [ ]:
LRS = [0.01, 0.1, 1.0, 20.0]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert set(lr_history) == set(LRS) and all(len(h) == 100 for h in lr_history.values()), "Нужны истории из 100 эпох для всех шагов"
last = {lr: (h[-1] if np.isfinite(h[-1]) else np.inf) for lr, h in lr_history.items()}
assert last[0.01] > last[1.0], "Малый шаг за 100 эпох должен дать заметно большие потери, чем шаг 1.0"
assert last[20.0] > last[1.0], "Слишком большой шаг должен дать худший результат (или разойтись)"
print("OK")

## Задание 3.3. Регуляризация на маленькой выборке

Возьмите только первые 45 объектов обучающей выборки (`X_train[:45]`, `y_train[:45]`) и большую сеть
`MLPClassifier(hidden_layer_sizes=(256, 256), solver="lbfgs", max_iter=5000, random_state=SEED, alpha=...)`.
Для каждого `alpha` из `ALPHAS` сохраните точность на этих 45 объектах (`reg_train`) и на **всей** тестовой выборке
(`reg_test`), постройте обе кривые (логарифмическая шкала по `alpha`).

In [ ]:
ALPHAS = [1e-5, 1e-3, 1e-1, 1.0, 3.0]
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert len(reg_train) == len(ALPHAS) and len(reg_test) == len(ALPHAS), "По значению на каждое alpha"
assert reg_train[0] == 1.0, "Без регуляризации большая сеть должна запомнить 45 объектов"
assert reg_test[-1] < max(reg_test), "При слишком сильной регуляризации качество на тесте должно падать"
assert reg_train[-1] < reg_train[0], "Сильная регуляризация должна мешать подгонке под обучающую выборку"
assert all(abs(v * 45 - round(v * 45)) < 1e-9 for v in reg_train), "reg_train — точность на тех же 45 объектах X_train[:45], y_train[:45]"
print("OK")

## Задание 3.4. Нулевая инициализация

Создайте параметры сети `[2, 64, 3]`, где **все** веса и смещения — нули (`np.zeros_like` от параметров
`init_params`), обучите её функцией `train` (шаг 0.5, 50 эпох) и сохраните точность на обучении в `zero_acc`, а
обученные параметры — в `zero_net`. Посмотрите на веса первого слоя `zero_net[0][0]`.

In [ ]:
# ВАШ КОД ЗДЕСЬ
raise NotImplementedError

In [ ]:
assert np.abs(zero_net[0][0]).max() == 0, "Веса первого слоя при нулевой инициализации должны остаться нулевыми"
assert abs(zero_acc - 1 / 3) < 0.05, f"Точность должна быть на уровне угадывания (около 1/3), получено {zero_acc:.3f}"
print("OK")

## Задание 3.5. Выводы

Ответьте на вопросы, опираясь на свои графики и числа. Ответ на каждый вопрос — 2–4 предложения.

1. Почему логистическая регрессия не справляется со спиралями, а сеть со скрытым слоем справляется? Что сеть
   делает с исходными признаками?
2. Как точность зависит от ширины скрытого слоя (задание 3.1)? Как это связано с теоремой об универсальной
   аппроксимации и с балансом смещения и разброса?
3. Опишите кривые обучения при разных шагах (задание 3.2). Почему малый шаг медленный, а слишком большой шаг
   приводит к росту потерь или `nan`?
4. Что показывает эксперимент с регуляризацией на 45 объектах (задание 3.3)? Как `alpha` влияет на точность на
   обучении и на тесте и почему?
5. Почему сеть с нулевой инициализацией не обучилась (задание 3.4)? Объясните по формулам обратного прохода, что
   происходит с градиентами весов. Что изменится, если инициализировать все веса одним и тем же ненулевым числом?
6. Зачем проверять градиент численно, если есть формулы? Почему для проверки берут маленькую сеть и несколько
   объектов, а для обучения численный градиент не используют?

*Ваш ответ:*